In [30]:
# ============================================
# 🎯 Complete Timestep-Aware Quantization Evaluation
# A comprehensive notebook for implementing and evaluating
# timestep-aware activation quantization on SDXL Lightning
# ============================================

# ============================================
# 📦 CELL 1: Imports & Environment Setup
# ============================================

import torch
import torch.nn as nn
from diffusers import StableDiffusionXLPipeline
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import time
import json
from pathlib import Path
import os

# Metrics
from torchmetrics.image.psnr import PeakSignalNoiseRatio
from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

print("✅ All imports successful!")

# Device setup
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("✅ Using Apple Silicon (MPS)")
elif torch.cuda.is_available():
    DEVICE = "cuda"
    print("✅ Using CUDA GPU")
else:
    DEVICE = "cpu"
    print("⚠️  Using CPU")


✅ All imports successful!
✅ Using Apple Silicon (MPS)


In [31]:
# ============================================
# 📦 CELL 2: Quantization Implementation
# ============================================

class ActivationQuantizer:
    """Quantize activations to specified bit-width."""
    
    @staticmethod
    def quantize_tensor(tensor, bits=8):
        """
        Quantize a tensor to N-bit representation.
        
        Args:
            tensor: Input tensor (float16/float32)
            bits: Target bit-width (4, 6, 8, 16)
        
        Returns:
            Quantized tensor in original dtype
        """
        if bits == 16:
            return tensor
        
        # Calculate quantization parameters
        qmin = 0
        qmax = 2 ** bits - 1
        
        # Get min/max of tensor
        min_val = tensor.min()
        max_val = tensor.max()
        
        # Avoid division by zero
        if max_val == min_val:
            return tensor
        
        # Calculate scale and zero point
        scale = (max_val - min_val) / qmax
        zero_point = qmin - min_val / scale
        
        # Quantize
        q_tensor = torch.clamp(
            torch.round(tensor / scale + zero_point),
            qmin, qmax
        )
        
        # Dequantize back to original range
        dq_tensor = (q_tensor - zero_point) * scale
        
        return dq_tensor.to(tensor.dtype)


class TimestepQuantizationSchedule:
    """Dynamic quantization schedule based on diffusion timestep."""
    
    def __init__(self, num_inference_steps=4, schedule_type="aggressive"):
        self.num_steps = num_inference_steps
        self.schedule_type = schedule_type
        self.current_step = 0
        
        # Define schedules
        self.schedules = {
            "aggressive": [4, 6, 8, 16],      # Max memory savings
            "balanced": [6, 8, 8, 16],        # Balance quality/memory
            "conservative": [8, 8, 16, 16],   # Prioritize quality
            "baseline": [16, 16, 16, 16],     # No quantization
        }
        
        self.schedule = self.schedules.get(schedule_type, self.schedules["balanced"])
        
        # Ensure schedule matches num_steps
        if len(self.schedule) != num_inference_steps:
            self.schedule = self._interpolate_schedule(num_inference_steps)
    
    def _interpolate_schedule(self, target_steps):
        """Interpolate schedule to match target number of steps."""
        original = np.array(self.schedule)
        x_old = np.linspace(0, 1, len(original))
        x_new = np.linspace(0, 1, target_steps)
        interpolated = np.interp(x_new, x_old, original)
        return [int(round(b)) for b in interpolated]
    
    def get_bits_for_step(self, step):
        """Get quantization bits for current step."""
        if step >= len(self.schedule):
            return 16
        return self.schedule[step]
    
    def reset(self):
        """Reset step counter."""
        self.current_step = 0


class QuantizationHook:
    """Forward hook to quantize linear layer activations."""
    
    def __init__(self, schedule: TimestepQuantizationSchedule, layer_name: str):
        self.schedule = schedule
        self.layer_name = layer_name
        self.quantizer = ActivationQuantizer()
    
    def __call__(self, module, input, output):
        """Hook function called during forward pass."""
        current_bits = self.schedule.get_bits_for_step(self.schedule.current_step)
        
        # Quantize output
        if isinstance(output, torch.Tensor):
            quantized = self.quantizer.quantize_tensor(output, bits=current_bits)
            return quantized
        elif isinstance(output, tuple):
            return tuple(
                self.quantizer.quantize_tensor(o, bits=current_bits) 
                if isinstance(o, torch.Tensor) else o 
                for o in output
            )
        
        return output


def apply_quantization_hooks(model, schedule: TimestepQuantizationSchedule, target_layers=None):
    """Apply quantization hooks to linear layers in the model."""
    hooks = []
    
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            if target_layers is not None:
                if not any(pattern in name for pattern in target_layers):
                    continue
            
            hook = QuantizationHook(schedule, name)
            handle = module.register_forward_hook(hook)
            hooks.append(handle)
    
    return hooks


def remove_quantization_hooks(hooks):
    """Remove all quantization hooks."""
    for handle in hooks:
        handle.remove()


class StepCallback:
    """Callback to track denoising steps and update quantization schedule."""
    
    def __init__(self, schedule):
        self.schedule = schedule
    
    def __call__(self, pipe, step_index, timestep, callback_kwargs):
        """Called at each denoising step."""
        self.schedule.current_step = step_index
        return callback_kwargs


def enable_quantization(pipe, schedule_type="balanced", num_inference_steps=4):
    """Enable timestep-aware quantization on a pipeline."""
    quantization_schedule = TimestepQuantizationSchedule(
        num_inference_steps=num_inference_steps,
        schedule_type=schedule_type
    )
    
    # Apply hooks to UNet
    quantization_hooks = apply_quantization_hooks(
        pipe.unet, 
        quantization_schedule,
        target_layers=["attn", "proj"]
    )
    
    # Create step callback
    step_callback = StepCallback(quantization_schedule)
    
    # Store references on the pipe object
    pipe._quantization_schedule = quantization_schedule
    pipe._quantization_hooks = quantization_hooks
    pipe._step_callback = step_callback
    
    return pipe


def disable_quantization(pipe):
    """Disable quantization hooks on a pipeline."""
    if hasattr(pipe, '_quantization_hooks') and pipe._quantization_hooks:
        remove_quantization_hooks(pipe._quantization_hooks)
        pipe._quantization_hooks = []
        pipe._quantization_schedule = None
        pipe._step_callback = None

print("✅ Quantization functions defined!")


✅ Quantization functions defined!


In [32]:
# ============================================
# 📦 CELL 3: Configuration
# ============================================

class Config:
    # Model path (adjust to your structure)
    MODEL_PATH = "../quantized_models/sdxl_lightning_4step_mps_fp16"
    
    # Output directories
    BASE_DIR = Path("./evaluation_outputs/timestep_quantization")
    BASELINE_DIR = BASE_DIR / "baseline_images"
    AGGRESSIVE_DIR = BASE_DIR / "aggressive_images"
    RESULTS_DIR = BASE_DIR / "results"
    
    # Test prompts
    TEST_PROMPTS = [
        "A serene mountain landscape at sunset with golden light",
        "A photorealistic portrait of a cat wearing glasses",
        "An astronaut riding a horse on Mars, digital art",
        "A cozy coffee shop interior with warm lighting",
        "Abstract geometric patterns in vibrant colors",
        "A futuristic city skyline at night with neon lights",
        "A close-up of a colorful butterfly on a flower",
        "Ancient temple ruins covered in vines and moss",
    ]
    
    # Generation settings
    IMAGE_SIZE = 512
    NUM_INFERENCE_STEPS = 4
    GUIDANCE_SCALE = 0.0
    SEED = 42

config = Config()

# Create directories
for d in [config.BASELINE_DIR, config.AGGRESSIVE_DIR, config.RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Configuration set!")
print(f"   Model: {config.MODEL_PATH}")
print(f"   Test prompts: {len(config.TEST_PROMPTS)}")
print(f"   Output: {config.BASE_DIR}")


✅ Configuration set!
   Model: ../quantized_models/sdxl_lightning_4step_mps_fp16
   Test prompts: 8
   Output: evaluation_outputs/timestep_quantization


In [33]:
# ============================================
# 📦 CELL 4: Image Generation Functions
# ============================================

def generate_baseline_images(prompts, output_dir):
    """Generate images with baseline FP16 model (no quantization)."""
    print("\n" + "="*70)
    print("🔵 BASELINE: Generating with FP16 (no activation quantization)")
    print("="*70)
    
    pipe = StableDiffusionXLPipeline.from_pretrained(
        config.MODEL_PATH,
        torch_dtype=torch.float16,
        use_safetensors=True,
        local_files_only=True,  # Force local loading
    )
    pipe = pipe.to(DEVICE)
    
    images = []
    times = []
    
    for i, prompt in enumerate(prompts):
        print(f"\n[{i+1}/{len(prompts)}] {prompt[:50]}...")
        
        generator = torch.Generator(device="cpu").manual_seed(config.SEED + i)
        
        start = time.time()
        image = pipe(
            prompt=prompt,
            num_inference_steps=config.NUM_INFERENCE_STEPS,
            guidance_scale=config.GUIDANCE_SCALE,
            generator=generator,
            height=config.IMAGE_SIZE,
            width=config.IMAGE_SIZE,
        ).images[0]
        elapsed = time.time() - start
        
        img_path = output_dir / f"baseline_{i:03d}.png"
        image.save(img_path)
        
        images.append(image)
        times.append(elapsed)
        print(f"  ✅ {elapsed:.2f}s")
    
    del pipe
    torch.mps.empty_cache() if DEVICE == "mps" else torch.cuda.empty_cache()
    
    print(f"\n📊 Average: {np.mean(times):.2f}s")
    return images, times


def generate_aggressive_images(prompts, output_dir):
    """Generate images with aggressive timestep-aware quantization."""
    print("\n" + "="*70)
    print("🟢 AGGRESSIVE: Generating with [4, 6, 8, 16]-bit schedule")
    print("="*70)
    
    pipe = StableDiffusionXLPipeline.from_pretrained(
        config.MODEL_PATH,
        torch_dtype=torch.float16,
        use_safetensors=True,
        local_files_only=True,  # Force local loading
    )
    pipe = pipe.to(DEVICE)
    
    pipe = enable_quantization(
        pipe,
        schedule_type="aggressive",
        num_inference_steps=config.NUM_INFERENCE_STEPS
    )
    
    images = []
    times = []
    
    for i, prompt in enumerate(prompts):
        print(f"\n[{i+1}/{len(prompts)}] {prompt[:50]}...")
        
        generator = torch.Generator(device="cpu").manual_seed(config.SEED + i)
        
        if hasattr(pipe, '_quantization_schedule'):
            pipe._quantization_schedule.reset()
        
        callback = pipe._step_callback if hasattr(pipe, '_step_callback') else None
        
        start = time.time()
        image = pipe(
            prompt=prompt,
            num_inference_steps=config.NUM_INFERENCE_STEPS,
            guidance_scale=config.GUIDANCE_SCALE,
            generator=generator,
            height=config.IMAGE_SIZE,
            width=config.IMAGE_SIZE,
            callback_on_step_end=callback,
        ).images[0]
        elapsed = time.time() - start
        
        img_path = output_dir / f"aggressive_{i:03d}.png"
        image.save(img_path)
        
        images.append(image)
        times.append(elapsed)
        print(f"  ✅ {elapsed:.2f}s")
    
    disable_quantization(pipe)
    del pipe
    torch.mps.empty_cache() if DEVICE == "mps" else torch.cuda.empty_cache()
    
    print(f"\n📊 Average: {np.mean(times):.2f}s")
    return images, times

print("✅ Image generation functions ready!")

✅ Image generation functions ready!


In [34]:
# ============================================
# 📦 CELL 5: Quality Metrics
# ============================================

def image_to_tensor(img):
    """Convert PIL image to tensor."""
    arr = np.array(img).astype(np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    return tensor


def compute_quality_metrics(baseline_images, quantized_images):
    """Compute PSNR, SSIM, and LPIPS metrics."""
    print("\n" + "="*70)
    print("📊 Computing Quality Metrics")
    print("="*70)
    
    psnr = PeakSignalNoiseRatio(data_range=1.0)
    ssim = StructuralSimilarityIndexMeasure(data_range=1.0)
    lpips = LearnedPerceptualImagePatchSimilarity(net_type='alex', normalize=True)
    
    results = {
        "per_image": [],
        "psnr": [],
        "ssim": [],
        "lpips": [],
    }
    
    for i, (baseline_img, quant_img) in enumerate(zip(baseline_images, quantized_images)):
        baseline_t = image_to_tensor(baseline_img)
        quant_t = image_to_tensor(quant_img)
        
        psnr_val = psnr(quant_t, baseline_t).item()
        ssim_val = ssim(quant_t, baseline_t).item()
        lpips_val = lpips(quant_t, baseline_t).item()
        
        results["psnr"].append(psnr_val)
        results["ssim"].append(ssim_val)
        results["lpips"].append(lpips_val)
        
        results["per_image"].append({
            "image_id": i,
            "psnr": psnr_val,
            "ssim": ssim_val,
            "lpips": lpips_val,
        })
        
        print(f"  Image {i}: PSNR={psnr_val:.2f}, SSIM={ssim_val:.4f}, LPIPS={lpips_val:.4f}")
    
    results["aggregate"] = {
        "psnr_mean": np.mean(results["psnr"]),
        "psnr_std": np.std(results["psnr"]),
        "ssim_mean": np.mean(results["ssim"]),
        "ssim_std": np.std(results["ssim"]),
        "lpips_mean": np.mean(results["lpips"]),
        "lpips_std": np.std(results["lpips"]),
    }
    
    return results

print("✅ Metrics functions ready!")


✅ Metrics functions ready!


In [35]:
# ============================================
# 📦 CELL 6: Results Analysis & Visualization
# ============================================

def print_summary(metrics, baseline_times, aggressive_times):
    """Print comprehensive evaluation summary."""
    agg = metrics["aggregate"]
    
    print("\n" + "="*70)
    print("📋 EVALUATION SUMMARY")
    print("="*70)
    
    print("\n⏱️  Performance:")
    baseline_avg = np.mean(baseline_times)
    aggressive_avg = np.mean(aggressive_times)
    speedup = baseline_avg / aggressive_avg
    
    print(f"   Baseline:    {baseline_avg:.2f}s/image")
    print(f"   Aggressive:  {aggressive_avg:.2f}s/image")
    print(f"   Speedup:     {speedup:.2f}x")
    
    print("\n📊 Image Quality:")
    print(f"   PSNR:  {agg['psnr_mean']:.2f} ± {agg['psnr_std']:.2f} dB")
    print(f"   SSIM:  {agg['ssim_mean']:.4f} ± {agg['ssim_std']:.4f}")
    print(f"   LPIPS: {agg['lpips_mean']:.4f} ± {agg['lpips_std']:.4f}")
    
    print("\n💡 Quality Assessment:")
    
    # Interpretations
    psnr_mean = agg['psnr_mean']
    if psnr_mean > 30:
        print("   ✅ PSNR: Excellent (minimal quality loss)")
    elif psnr_mean > 25:
        print("   ✅ PSNR: Good (acceptable quality)")
    elif psnr_mean > 20:
        print("   ⚠️  PSNR: Fair (noticeable degradation)")
    else:
        print("   ❌ PSNR: Poor (significant degradation)")
    
    ssim_mean = agg['ssim_mean']
    if ssim_mean > 0.95:
        print("   ✅ SSIM: Excellent (structurally very similar)")
    elif ssim_mean > 0.90:
        print("   ✅ SSIM: Good (structurally similar)")
    elif ssim_mean > 0.80:
        print("   ⚠️  SSIM: Fair (some differences)")
    else:
        print("   ❌ SSIM: Poor (significant differences)")
    
    lpips_mean = agg['lpips_mean']
    if lpips_mean < 0.1:
        print("   ✅ LPIPS: Excellent (perceptually very similar)")
    elif lpips_mean < 0.2:
        print("   ✅ LPIPS: Good (perceptually similar)")
    elif lpips_mean < 0.3:
        print("   ⚠️  LPIPS: Fair (noticeable differences)")
    else:
        print("   ❌ LPIPS: Poor (significant differences)")
    
    print("\n🎯 Recommendation:")
    if psnr_mean > 25 and ssim_mean > 0.90 and lpips_mean < 0.2:
        print("   ✅ Aggressive quantization is RECOMMENDED")
        print("   ✅ Minimal quality loss with significant speedup")
    elif psnr_mean > 20 and ssim_mean > 0.80:
        print("   ⚠️  Moderate quality trade-off")
        print("   ⚠️  Consider for non-critical applications")
    else:
        print("   ❌ Quality loss is significant")
        print("   ❌ Try more conservative schedule")


def create_comparison_grid(baseline_images, aggressive_images, num_samples=4):
    """Create side-by-side comparison grid."""
    num_samples = min(num_samples, len(baseline_images))
    
    img_size = config.IMAGE_SIZE
    grid_width = img_size * num_samples
    grid_height = img_size * 2 + 60
    
    grid = Image.new('RGB', (grid_width, grid_height), color='white')
    draw = ImageDraw.Draw(grid)
    
    for i in range(num_samples):
        grid.paste(baseline_images[i], (i * img_size, 30))
        grid.paste(aggressive_images[i], (i * img_size, img_size + 30))
    
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 20)
    except:
        font = ImageFont.load_default()
    
    draw.text((10, 5), "Baseline (FP16)", fill='black', font=font)
    draw.text((10, img_size + 35), "Aggressive [4,6,8,16]", fill='black', font=font)
    
    grid_path = config.RESULTS_DIR / "comparison_grid.png"
    grid.save(grid_path)
    print(f"\n🖼️  Comparison grid: {grid_path}")
    
    return grid


def save_results(metrics, baseline_times, aggressive_times):
    """Save results to JSON."""
    results = {
        "config": {
            "model": config.MODEL_PATH,
            "num_prompts": len(config.TEST_PROMPTS),
            "image_size": config.IMAGE_SIZE,
        },
        "performance": {
            "baseline_avg": float(np.mean(baseline_times)),
            "aggressive_avg": float(np.mean(aggressive_times)),
            "speedup": float(np.mean(baseline_times) / np.mean(aggressive_times)),
        },
        "quality_metrics": metrics,
        "schedule": {
            "baseline": [16, 16, 16, 16],
            "aggressive": [4, 6, 8, 16],
        },
    }
    
    path = config.RESULTS_DIR / "evaluation_results.json"
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"💾 Results saved: {path}")

print("✅ Analysis functions ready!")


✅ Analysis functions ready!


In [36]:
# ============================================
# 📦 CELL 7: RUN EVALUATION
# ============================================

def run_evaluation():
    """Run complete evaluation pipeline."""
    print("\n" + "="*70)
    print("🚀 STARTING TIMESTEP QUANTIZATION EVALUATION")
    print("="*70)
    
    # Generate baseline
    baseline_images, baseline_times = generate_baseline_images(
        config.TEST_PROMPTS,
        config.BASELINE_DIR
    )
    
    # Generate aggressive
    aggressive_images, aggressive_times = generate_aggressive_images(
        config.TEST_PROMPTS,
        config.AGGRESSIVE_DIR
    )
    
    # Compute metrics
    metrics = compute_quality_metrics(baseline_images, aggressive_images)
    
    # Print summary
    print_summary(metrics, baseline_times, aggressive_times)
    
    # Create visualization
    comparison_grid = create_comparison_grid(baseline_images, aggressive_images)
    
    # Save results
    save_results(metrics, baseline_times, aggressive_times)
    
    print("\n✅ EVALUATION COMPLETE!")
    
    return {
        "baseline_images": baseline_images,
        "aggressive_images": aggressive_images,
        "metrics": metrics,
        "baseline_times": baseline_times,
        "aggressive_times": aggressive_times,
    }

# Run it!
results = run_evaluation()


🚀 STARTING TIMESTEP QUANTIZATION EVALUATION

🔵 BASELINE: Generating with FP16 (no activation quantization)


Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00,  8.75it/s]



[1/8] A serene mountain landscape at sunset with golden ...


100%|██████████| 4/4 [00:05<00:00,  1.35s/it]


  ✅ 14.20s

[2/8] A photorealistic portrait of a cat wearing glasses...


...

  ✅ 47.84s

[7/8] A close-up of a colorful butterfly on a flower...


100%|██████████| 4/4 [00:23<00:00,  5.98s/it]


  ✅ 42.58s

[8/8] Ancient temple ruins covered in vines and moss...


100%|██████████| 4/4 [00:26<00:00,  6.62s/it]


  ✅ 43.09s

📊 Average: 28.54s

📊 Computing Quality Metrics
  Image 0: PSNR=12.37, SSIM=0.5291, LPIPS=0.6365
  Image 1: PSNR=12.02, SSIM=0.5129, LPIPS=0.4930
  Image 2: PSNR=9.18, SSIM=0.2617, LPIPS=0.7193
  Image 3: PSNR=10.52, SSIM=0.3530, LPIPS=0.6930
  Image 4: PSNR=9.73, SSIM=0.2653, LPIPS=0.7002
  Image 5: PSNR=9.54, SSIM=0.2432, LPIPS=0.6817
  Image 6: PSNR=10.62, SSIM=0.4318, LPIPS=0.6516
  Image 7: PSNR=12.45, SSIM=0.2623, LPIPS=0.6933

📋 EVALUATION SUMMARY

⏱️  Performance:
   Baseline:    13.25s/image
   Aggressive:  28.54s/image
   Speedup:     0.46x

📊 Image Quality:
   PSNR:  10.80 ± 1.23 dB
   SSIM:  0.3574 ± 0.1112
   LPIPS: 0.6586 ± 0.0673

💡 Quality Assessment:
   ❌ PSNR: Poor (significant degradation)
   ❌ SSIM: Poor (significant differences)
   ❌ LPIPS: Poor (significant differences)

🎯 Recommendation:
   ❌ Quality loss is significant
   ❌ Try more conservative schedule

🖼️  Comparison grid: evaluation_outputs/timestep_quantization/results/comparison_grid.png
💾 Resul

In [29]:
import os
from pathlib import Path

# Show where we are
current_dir = Path.cwd()
print(f"Current directory: {current_dir}")

# List parent directories
print(f"\nParent: {current_dir.parent}")
print(f"Grandparent: {current_dir.parent.parent}")

# Try to find the model
possible_paths = [
    "quantized_models/sdxl_lightning_4step_mps_fp16",
    "../quantized_models/sdxl_lightning_4step_mps_fp16",
    "../../quantized_models/sdxl_lightning_4step_mps_fp16",
]

print("\n🔍 Checking paths:")
for p in possible_paths:
    exists = Path(p).exists()
    abs_path = Path(p).absolute()
    print(f"  {p}")
    print(f"    → {abs_path}")
    print(f"    → {'✅ EXISTS' if exists else '❌ NOT FOUND'}")

Current directory: /Users/yash.honrao/Documents/Fall 25/GenAI/Project/sdxl/timestep aware quantization

Parent: /Users/yash.honrao/Documents/Fall 25/GenAI/Project/sdxl
Grandparent: /Users/yash.honrao/Documents/Fall 25/GenAI/Project

🔍 Checking paths:
  quantized_models/sdxl_lightning_4step_mps_fp16
    → /Users/yash.honrao/Documents/Fall 25/GenAI/Project/sdxl/timestep aware quantization/quantized_models/sdxl_lightning_4step_mps_fp16
    → ❌ NOT FOUND
  ../quantized_models/sdxl_lightning_4step_mps_fp16
    → /Users/yash.honrao/Documents/Fall 25/GenAI/Project/sdxl/timestep aware quantization/../quantized_models/sdxl_lightning_4step_mps_fp16
    → ✅ EXISTS
  ../../quantized_models/sdxl_lightning_4step_mps_fp16
    → /Users/yash.honrao/Documents/Fall 25/GenAI/Project/sdxl/timestep aware quantization/../../quantized_models/sdxl_lightning_4step_mps_fp16
    → ❌ NOT FOUND
